# Spatial operations

In this notebook we will do spatial operations using DuckDB and Python.

<a target="_blank" href="https://colab.research.google.com/github/kraina-ai/srai-tutorial/blob/dawts2026/02_spatial_operations.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

In [ ]:
## Uncomment the line below when running on Google Colab
# %pip install geopandas pooch folium branca mapclassify h3 shapely

## Transactions data analysis - RCN dataset for Warsaw

We will use public data from deweloperuch github repository and aggregate it into a heatmap using H3 cells.

In [ ]:
from pooch import retrieve, Unzip

In [ ]:
RCN_WARSAW_URL = 'https://github.com/deweloperuch/rejestr-cen-nieruchomosci/raw/refs/heads/main/data/warszawa/warszawa_rcn.zip'

rcn_file_path = retrieve(RCN_WARSAW_URL, path='files/rcn', known_hash=None, processor=Unzip())[0]

In [ ]:
import geopandas as gpd

gpd.list_layers(rcn_file_path)

In [ ]:
flats = gpd.read_file(rcn_file_path, layer="RCN_Lokal").rename(
    columns={"powUzytkowaLokalu": "area_m2", "cenaLokaluBrutto": "price"}
)
flats

In [ ]:
flats.plot(markersize=5)

In [ ]:
flats["price_per_m2"] = (flats["price"] / flats["area_m2"]).round(2)
flats

In [ ]:
flats["price_per_m2"].describe(
    [0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
).round(2)

In [ ]:
flats_filtered = flats[
    flats["price_per_m2"].between(
        flats["price_per_m2"].quantile(0.01), flats["price_per_m2"].quantile(0.99)
    )
].copy()
flats_filtered["price_per_m2"].hist()

In [ ]:
flats_filtered.plot("price_per_m2", legend=True)

In [ ]:
import h3


def point_to_h3(point):
    lon = point.x
    lat = point.y
    return h3.latlng_to_cell(lat, lon, 8)


flats_filtered["h3"] = flats_filtered.geometry.to_crs(4326).apply(point_to_h3)
flats_filtered

In [ ]:
aggregated_flats = (
    flats_filtered.groupby("h3")
    .agg(
        price_per_m2=("price_per_m2", "median"),
        total_transactions=("gml_id", "nunique"),
    )
    .round(2)
    .reset_index()
)
aggregated_flats

In [ ]:
from shapely import Polygon


def h3_to_geometry(h3_id):
    coords = h3.cell_to_boundary(h3_id)
    flipped = tuple(coord[::-1] for coord in coords)
    return Polygon(flipped)


aggregated_flats = gpd.GeoDataFrame(
    aggregated_flats, geometry=aggregated_flats["h3"].apply(h3_to_geometry), crs=4326
)
aggregated_flats

In [ ]:
aggregated_flats.explore(
    "price_per_m2",
    cmap="RdYlGn_r",
    tiles="CartoDB Voyager",
    style_kwds=dict(color="black", weight=0.5),
)